# 🛡️ GUARDIAN Fleet — OpenEnv Hackathon Training Notebook

**Multi-agent AI security oversight environment trained via GRPO**

- Theme: #1 Multi-Agent + #3 World Modeling + Fleet AI (Scalable Oversight)
- Model: Qwen2.5-7B (Unsloth 4-bit LoRA)
- Training: TRL GRPOTrainer
- Environment: GUARDIAN OpenEnv Gymnasium wrapper

**Architecture:**
```
Worker Fleet (Finance/Ops/HR) ←→ Enterprise Graph DB
         ↓ telemetry (partial observability)
  Guardian Agent (Qwen2.5-7B, GRPO-trained)
         ↓ intervention (11 types)
  Compliance Simulator (GPT-4o-mini, expert-in-the-loop)
         ↓ approval/denial
  Curriculum Agent (GPT-4o-mini, generates harder attacks)
         ↑ UCB Attack Selector (exploration vs exploitation)
```


In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-500:]}')
    else:
        print(f'✓ {cmd[:60]}')
    return result

run('pip install "unsloth[kaggle-new]" --upgrade -q')
run('pip install openenv-core[cli] -q')
run('pip install trl>=0.8.6 peft datasets openai python-dotenv gradio -q')
run('pip install gymnasium -q')
print('✅ All dependencies installed')

In [ ]:
# ── Step 2: Clone / upload GUARDIAN code ─────────────────────────────────────
import os

# If running from Kaggle with dataset attached:
# !cp -r /kaggle/input/guardian-env/* .

# Or clone from HuggingFace/GitHub:
# run('git clone https://github.com/YOUR_TEAM/guardian-env .')

# For now, create minimal structure inline:
os.makedirs('guardian/data', exist_ok=True)
os.makedirs('guardian/checkpoints', exist_ok=True)
os.makedirs('guardian/environment', exist_ok=True)
os.makedirs('guardian/agents', exist_ok=True)
os.makedirs('guardian/training', exist_ok=True)

# Set OPENAI_API_KEY from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    print('✅ API key loaded from Kaggle secrets')
except Exception:
    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', '')
    if os.environ['OPENAI_API_KEY']:
        print('✅ API key from environment')
    else:
        print('⚠️  No API key found — Worker will use fallback mode')

In [ ]:
# ── Step 3: OpenEnv setup ─────────────────────────────────────────────────────
# Verify openenv-core is installed
import openenv_core
print(f'openenv-core version: {openenv_core.__version__}')

# Register GUARDIAN as a gymnasium environment
import sys
sys.path.insert(0, '.')

import gymnasium as gym
import guardian.environment.openenv_wrapper  # registers Guardian-v0
env = gym.make('Guardian-v0')
obs, info = env.reset(seed=42)
print(f'✅ Guardian-v0 registered and reset: obs keys={list(obs.keys())}')
env.close()

In [ ]:
# ── Step 4: Load Qwen2.5-7B with Unsloth ─────────────────────────────────────
from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model

print('Loading Qwen2.5-7B-Instruct (4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()
print('✅ Model ready')

In [ ]:
# ── Step 5: Initialise agent fleet ────────────────────────────────────────────
from guardian.environment.guardian_env import GUARDIANEnvironment
from guardian.environment.reward_computer import RewardComputer
from guardian.agents.worker_agent import WorkerAgent
from guardian.agents.guardian_agent import GuardianAgent
from guardian.agents.compliance_simulator import ComplianceSimulator
from guardian.agents.curriculum_agent import UCBAttackSelector, CurriculumAgent
from guardian.training.episode_runner import EpisodeRunner

api_key = os.environ.get('OPENAI_API_KEY', '')

ATTACK_POOL = [
    None,
    'authority_spoofing',
    'prompt_injection',
    'approval_bypass',
    'data_exfiltration',
    'confused_deputy',
    'approval_laundering',
]

worker = WorkerAgent(role='finance', api_key=api_key)
guardian_agent = GuardianAgent()
guardian_agent.model = model
guardian_agent.tokenizer = tokenizer

ucb = UCBAttackSelector(attack_pool=ATTACK_POOL)
curriculum = CurriculumAgent(api_key=api_key)
compliance = ComplianceSimulator(api_key=api_key)

runner = EpisodeRunner(
    env=GUARDIANEnvironment(),
    worker=worker,
    guardian=guardian_agent,
    reward_computer=RewardComputer(),
    compliance_sim=compliance,
    curriculum_agent=curriculum,
    ucb_selector=ucb,
)
runner._use_ucb = True
print('✅ Agent fleet ready')

In [ ]:
# ── Step 6: Sanity check — 5 episodes before training ─────────────────────────
import json, random

print('Sanity check: 5 episodes...')
for i in range(5):
    atk = random.choice(ATTACK_POOL)
    result = runner.run_episode(attack_type=atk)
    print(f'  ep={i+1} | {str(atk):<22} | intact={result.production_intact} | '
          f'fork={result.fork_triggered} | reward={result.reward:.3f}')

print('✅ Sanity check passed')

In [ ]:
# ── Step 7: Baseline measurement ─────────────────────────────────────────────
print('Recording baseline (20 episodes)...')
baseline_results = []

for i in range(20):
    atk = random.choice(ATTACK_POOL)
    try:
        result = runner.run_episode(attack_type=atk)
        baseline_results.append({
            'attack_type': atk,
            'reward': result.reward,
            'production_intact': result.production_intact,
            'fork_triggered': result.fork_triggered,
            'detected': result.guardian_detected_type,
        })
    except Exception as e:
        print(f'  Error ep {i}: {e}')

os.makedirs('guardian/data', exist_ok=True)
with open('guardian/data/baseline_untrained.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

mean_r = sum(b['reward'] for b in baseline_results) / len(baseline_results)
detect_rate = sum(1 for b in baseline_results if b['detected'] and b['attack_type']) / max(1, sum(1 for b in baseline_results if b['attack_type']))
print(f'✅ Baseline: mean_reward={mean_r:.4f}, detect_rate={detect_rate:.1%}')

In [ ]:
# ── Step 8: GRPO Training Loop ────────────────────────────────────────────────
import gc, time
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

TOTAL_EPISODES = 100   # increase to 200 on Kaggle T4 x2
TRAIN_EVERY = 8
SAVE_EVERY = 50
LOG_FILE = 'guardian/data/training_log.jsonl'
SCORECARD_FILE = 'guardian/data/scorecards.jsonl'

grpo_config = GRPOConfig(
    output_dir='guardian/checkpoints/grpo_tmp',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    max_steps=8,
    warmup_steps=2,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
    gradient_checkpointing=True,
    fp16=True,
    optim='adamw_8bit',
)

all_samples = []
samples_map = {}
reward_curve = []  # for plotting

print(f'Training {TOTAL_EPISODES} episodes, GRPO every {TRAIN_EVERY}\n')

for ep in range(1, TOTAL_EPISODES + 1):
    t0 = time.time()
    try:
        result = runner.run_episode()
    except Exception as e:
        print(f'  ep={ep:03d} ERROR: {e}')
        continue

    elapsed = time.time() - t0
    reward_curve.append({'ep': ep, 'reward': result.reward, 'attack': result.attack_type})

    # Log
    entry = {
        'episode': ep, 'attack_type': result.attack_type,
        'production_intact': result.production_intact,
        'fork_triggered': result.fork_triggered,
        'reward': round(result.reward, 4), 'elapsed_s': round(elapsed, 1),
        'detected': result.guardian_detected_type,
    }
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    with open(SCORECARD_FILE, 'a') as f:
        f.write(json.dumps(result.scorecard) + '\n')

    if ep % 10 == 0:
        recent_rewards = [r['reward'] for r in reward_curve[-10:]]
        mean_r = sum(recent_rewards) / len(recent_rewards)
        print(f'  ep={ep:03d} | {str(result.attack_type):<22} | '
              f'reward={result.reward:.3f} | mean10={mean_r:.3f} | {elapsed:.1f}s')

    for s in result.training_samples:
        all_samples.append(s)
        samples_map[s['prompt'][:100]] = result.reward

    torch.cuda.empty_cache()
    gc.collect()

    # GRPO training
    if ep % TRAIN_EVERY == 0 and len(all_samples) >= 4:
        print(f'  >>> GRPO training on {len(all_samples)} samples...')
        dataset = Dataset.from_list([{'prompt': s['prompt']} for s in all_samples])

        def reward_fn(prompts, completions, **kwargs):
            return [samples_map.get(p[:100], 0.5) for p in prompts]

        try:
            GRPOTrainer(
                model=model,
                reward_funcs=[reward_fn],
                args=grpo_config,
                train_dataset=dataset,
                processing_class=tokenizer,
            ).train()
            print('  >>> Done.')
        except Exception as e:
            print(f'  >>> Training error (continuing): {e}')

        all_samples = []
        samples_map = {}
        torch.cuda.empty_cache()
        gc.collect()

    if ep % SAVE_EVERY == 0:
        ckpt = f'guardian/checkpoints/episode_{ep}'
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f'  >>> Checkpoint: {ckpt}')

print('\n✅ Training complete!')

In [ ]:
# ── Step 9: Reward curve plot ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

episodes = [r['ep'] for r in reward_curve]
rewards = [r['reward'] for r in reward_curve]

# Smooth with rolling average
window = 10
smooth = [np.mean(rewards[max(0,i-window):i+1]) for i in range(len(rewards))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
axes[0].plot(episodes, rewards, alpha=0.3, color='steelblue', linewidth=0.8, label='Episode reward')
axes[0].plot(episodes, smooth, color='steelblue', linewidth=2, label=f'Smoothed (window={window})')
axes[0].axhline(y=np.mean(rewards[:20]), color='red', linestyle='--', alpha=0.5, label='Baseline mean')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[0].set_title('GUARDIAN GRPO Reward Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Attack distribution
from collections import Counter
attack_counts = Counter(str(r['attack']) for r in reward_curve)
labels = list(attack_counts.keys())
sizes = list(attack_counts.values())
axes[1].bar(range(len(labels)), sizes, color='steelblue')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=45, ha='right')
axes[1].set_title('UCB Attack Distribution')
axes[1].set_ylabel('Episode count')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('guardian/data/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: guardian/data/reward_curve.png')

In [ ]:
# ── Step 10: Evaluation — 4 headline numbers ──────────────────────────────────
from guardian.training.evaluation import EvaluationHarness

harness = EvaluationHarness(
    scorecard_file=SCORECARD_FILE,
    baseline_file='guardian/data/baseline_untrained.json',
)
harness.print_report()

In [ ]:
# ── Step 11: Save final model ─────────────────────────────────────────────────
final_dir = 'guardian/checkpoints/final'
os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'✅ Final model saved to {final_dir}')

# Optional: push to HuggingFace Hub
# model.push_to_hub('your-org/guardian-fleet-qwen2.5-7b-grpo')
# tokenizer.push_to_hub('your-org/guardian-fleet-qwen2.5-7b-grpo')

In [ ]:
# ── Step 12: Calibration Plot (Reliability Diagram) ───────────────────────────
import json

scorecards = harness.load_scorecards()
metrics = harness.compute_metrics(scorecards)
cal_bins = metrics.get('calibration_bins', [])

if cal_bins:
    fig, ax = plt.subplots(figsize=(7, 7))
    predicted = [b['mean_predicted'] for b in cal_bins]
    actual = [b['actual_attack_rate'] for b in cal_bins]
    sizes = [b['n'] * 20 for b in cal_bins]

    ax.scatter(predicted, actual, s=sizes, color='steelblue', alpha=0.7, zorder=3)
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect calibration (y=x)')
    ax.set_xlabel('Mean Predicted Risk Score')
    ax.set_ylabel('Actual Attack Rate')
    ax.set_title('GUARDIAN Calibration — Reliability Diagram')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('guardian/data/calibration_plot.png', dpi=150)
    plt.show()
    print('✅ Calibration plot saved')
else:
    print('Not enough data for calibration plot')

In [ ]:
# ── Step 13: UCB Attack Stats ─────────────────────────────────────────────────
print('\n=== UCB Attack Selector Statistics ===')
for atk, stats in ucb.get_stats().items():
    n = stats['count']
    mr = stats['mean_reward']
    if n and mr is not None:
        bar = '█' * int((1-mr) * 10) + '░' * (10 - int((1-mr) * 10))
        print(f'  {str(atk):<25} | n={n:>3} | mean_r={mr:.3f} | weakness [{bar}]')

weakness = ucb.detection_weakness()
print(f'\n  Guardian weakest against: {weakness}')
print('\n✅ Training notebook complete!')